# Prerequisites: Creating Sample Agents

## Overview

Let's first start by creating agents to be evaluated. This tutorial creates two sample agents for evaluation using different frameworks:
- [Strands Agents SDK](https://strandsagents.com/)
- [LangGraph](https://www.langchain.com/langgraph)

Both agents uses Anthropic Claude Haiku 4.5 from Amazon Bedrock as the LLM model but you can use any model of your preference and they have identical capabilities:
- **Math Tool**: Tool to perform basic math operations
- **Weather Tool**: Dummy implementation for weather tool


The architecture looks as following:

![Architecture](../images/agent_architecture.png)

## Prerequisites
- Python 3.10+
- AWS credentials

In [ ]:
!pip install -r ../requirements.txt -q

## Deploy Agents

The `deploy_agents.py` script handles all infrastructure setup for both agents:
- Creates the AgentCore project scaffolds (if not already present)
- Copies the agent implementations into each project
- Deploys both agents to AgentCore Runtime
- Waits until both reach **READY** status
- Saves the agent IDs and ARNs to `agents_config.json`

Running the cell below will take **~5 minutes** on the first run. On subsequent runs the script detects the already-deployed agents and exits quickly.

In [ ]:
!python deploy_agents.py

## Load Agent Configuration

Read the agent IDs and ARNs written by `deploy_agents.py`.

In [ ]:
import json
import uuid
import boto3

with open("agents_config.json") as f:
    _cfg = json.load(f)

region = _cfg["region"]
agent_id_strands = _cfg["strands"]["agent_id"]
agent_arn_strands = _cfg["strands"]["agent_arn"]
agent_id_langgraph = _cfg["langgraph"]["agent_id"]
agent_arn_langgraph = _cfg["langgraph"]["agent_arn"]

print(f"Region    : {region}")
print(f"Strands   : {agent_id_strands}")
print(f"LangGraph : {agent_id_langgraph}")

## Invoke the Strands Agent

Let's test the Strands agent by invoking the AgentCore Runtime endpoint using the `bedrock-agentcore` boto3 client.

When an agent session is created, AgentCore assigns it a unique `runtimeSessionId`. All subsequent turns in the same conversation pass the same session ID so the agent can maintain context across turns. The response is streamed back as a sequence of event chunks.

> **Learn more:** [Invoke an AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/invoke-agent-runtime.html)


In [ ]:
session_id_strands = str(uuid.uuid4())
print(f"Strands session ID: {session_id_strands}")

In [ ]:
agentcore_dp = boto3.client("bedrock-agentcore", region_name=region)


def invoke_agent(agent_arn, prompt, session_id):
    """Invoke an AgentCore Runtime agent and return its text response."""
    response = agentcore_dp.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}).encode("utf-8"),
    )
    raw = response["response"].read().decode("utf-8")
    parts = []
    for line in raw.splitlines():
        if line.startswith("data: "):
            chunk = line[len("data: ") :]
            try:
                chunk = json.loads(chunk)
            except Exception:
                pass
            parts.append(str(chunk))
    return "".join(parts) if parts else raw


result = invoke_agent(agent_arn_strands, "How much is 2+2?", session_id_strands)
print(result)

In [ ]:
result = invoke_agent(agent_arn_strands, "How is the weather now?", session_id_strands)
print(result)

In [ ]:
result = invoke_agent(
    agent_arn_strands, "Can you tell me the capital of the US?", session_id_strands
)
print(result)

## Invoke the LangGraph Agent

The LangGraph agent was deployed alongside the Strands agent by `deploy_agents.py`. Let's test it.

Test the LangGraph agent with the same questions:

In [ ]:
session_id_langgraph = str(uuid.uuid4())
print(f"LangGraph session ID: {session_id_langgraph}")

In [ ]:
result = invoke_agent(agent_arn_langgraph, "What is 2+2?", session_id_langgraph)
print(result)

In [ ]:
result = invoke_agent(
    agent_arn_langgraph, "What is the weather now?", session_id_langgraph
)
print(result)

In [ ]:
result = invoke_agent(
    agent_arn_langgraph, "Can you tell me the capital of the US?", session_id_langgraph
)
print(result)

In [ ]:
print(f"Strands   agent_id={agent_id_strands}")
print(f"          agent_arn={agent_arn_strands}")
print(f"          session={session_id_strands}")
print(f"LangGraph agent_id={agent_id_langgraph}")
print(f"          agent_arn={agent_arn_langgraph}")
print(f"          session={session_id_langgraph}")

In [ ]:
%store agent_id_strands
%store agent_arn_strands
%store session_id_strands
%store agent_id_langgraph
%store agent_arn_langgraph
%store session_id_langgraph

## Next Steps

Now that you have all the required pre-requisites, let's go through the individual evaluation tutorials:
Continue with the evaluation tutorials:
- [01-creating-custom-evaluators](../01-creating-custom-evaluators/): Create custom evaluators
- [02-running-evaluations](../02-running-evaluations/): Run on-demand and online evaluations
- [03-evaluation-workflows](../03-evaluation-workflows/): : Advanced techniques and dashboards